# Introduction to Landlab Toolkit

*CU Boulder GEOL 3600 Introduction to Python Programming for Geoscientists, November 2025*

## What is Landlab?

[Landlab](https://landlab.github.io) is a Python package designed to make it easier to build grid-based numerical models. Among other things, Landlab provides:

1. **Grid objects** that contain data and methods to manage a 2D grid
2. **Fields:** arrays of data that can be added to grid elements
3. **Components:** Python classes that implement one particular process or function
3. **Utilities:** various helpful functions to making calculations, reading input, and writing output

### Making a grid

Here we'll make a small `RasterModelGrid` and play with it.
```
from landlab import RasterModelGrid

nrows = 5 # number of node rows
ncols = 4 # number of node columns
dx = 10.0 # spacing between nodes

# Instantiate a new RasterModelGrid object
grid = RasterModelGrid((nrows, ncols), dx)

# Report the number of nodes
print(grid.number_of_nodes)

# Plot the node positions
plt.scatter(grid.x_of_node, grid.y_of_node)
```

In [ ]:
from landlab import RasterModelGrid

nrows = 5 # number of node rows
ncols = 4 # number of node columns
dx = 10.0 # spacing between nodes

# Instantiate a new RasterModelGrid object
grid = RasterModelGrid((nrows, ncols), dx)

# Report the number of nodes
print(grid.number_of_nodes)

# Plot the node positions
plt.scatter(grid.x_of_node, grid.y_of_node)

### Inspecting grid elements with **grid sketchbook**

The online **[grid sketchbook](https://landlab.github.io/grid-sketchbook)** helps visualize the different elements of a Landlab grid.

Go to https://landlab.github.io/grid-sketchbook and create a Raster grid with 4 rows and 5 columns.

Notice the six element types: nodes, links, patches, corners, faces, and cells.

#### Questions

1. Which link connects nodes 6 and 7?
2. Which node is inside of cell 4?
3. Which cell contains node 8?
4. Which node does link 14 point to?


## A 2D diffusion model in Landlab

The following demonstrates how we can use a Landlab grid to implement our earlier hillslope simulation, but this time in 2D instead of 1D. The point here is to give you a first glimpse at how you can code up a finite-difference model using a Landlab grid.

In [ ]:
# Make a grid
grid = RasterModelGrid((16, 10), 10.0)

# Make an array to hold the elevation values, initially at 100 m
z = np.zeros(grid.number_of_nodes) + 100.0 # one elevation value per grid node

# Parameters
num_steps = 20 # number of time steps
dt = 1000.0 # size of a time step in years
lowering_rate = 0.001 # lowering rate of boundaries, m/y
drop_per_step = lowering_rate * dt # elevation drop of grid edges each time step, in meters
Kc = 0.01 # creep coefficient, m2/y

# Time loop
for _ in range(num_steps):

    # lower the boundary nodes
    z[grid.perimeter_nodes] -= drop_per_step

    # calculate gradients at links (Landlab's 2D version of "diff(z) / dx"!)
    grad = grid.calc_grad_at_link(z) 

    # calculate soil flux at each link
    q = -Kc * grad

    # calculate ins and outs of soil in each cell (2D version of "diff(q) / dx")
    dqdx = grid.calc_flux_div_at_node(q) 

    # update elevations
    z[grid.core_nodes] -= dqdx[grid.core_nodes] * dt 

Now let's make a surface plot of the resulting landform.

(Note: Landlab arrays are 1D, but the matplotlib `plot_surface()` function wants 2D arrays, so we have to reshape them before plotting.)

In [ ]:
# "turn on" 3D plotting
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# plot_surface() wants 2d arrays, so we have to reshape our Landlab arrays
nrows = grid.number_of_node_rows
ncols = grid.number_of_node_columns
x2d = grid.x_of_node.reshape((nrows, ncols))
y2d = grid.y_of_node.reshape((nrows, ncols))
z2d = z.reshape((nrows, ncols))

# make a surface plot
ax.plot_surface(x2d, y2d, z2d)
ax.set_xlabel('East-west distance (m)')
ax.set_ylabel('North-south distance (m)')
ax.set_zlabel('Elevation (m)')

### <div style="color:green">Optional: playing with the model</div>

Experiment with your model.

1. What happens if you run it longer? Does the height of the hill (relative to the base) keep rising, or does it reach a steady state?

2. What happens if you increase the creep coefficient by 4x? (e.g., wetter climate)

3. What happens if you lower the creep coefficient by 4x? (e.g., drier climate)

4. What happens if you increase the lowering rate?

5. What happens if you reduce the lowering rate?

6. What happens if you make the model domain (i.e., the size of the hill) bigger or smaller?

### *Optional: advanced challenge problems*

If the finite-difference diffusion solution seems straightforward, try the following:

1. Combine the two finite-difference equations for 1D diffusion together to show that

$$z_i^{t+1} = z_i^t + \frac{K_c\Delta t}{\Delta x^2} \left(z_{i+1}^t - 2 z_i^t + z_{i-1}^t \right)$$

2. Write a program that implements this solution directly. Hint: think about what the following numpy array operation does:

```
z[2:] - 2 * z[1:-1] + z[:-2]
```

3. Modify your program so that the $K_c$, $\Delta t$, and $\Delta x^2$ are combined in a single variable called `alpha`. Experiment with different values of this parameter. What does your calculation look like when `alpha = 0.2`? When `alpha = 1`? Look back to how we calculated time-step size in Notebook 17.

4. In the Landlab-based 2D model, try making `Kc` vary in space. (Hint: you'll need to make `Kc` an array, with one value per grid link.)